**How would we translate a query like this?**
``` sql
SELECT max(b) 
FROM stuff
```

**Create some fake data**


In [1]:
from faker import Faker

fake = Faker()
fake.seed_instance(42)

from dataclasses import dataclass

number_of_tuples = 40


@dataclass
class Stuff:
    a: int
    b: int


# create some fake data mimicking tuples in a relation:
data = [Stuff(fake.pyint(max_value=1000), fake.pyint(max_value=2000)) for f in range(number_of_tuples)]
data[:10]

[Stuff(a=654, b=228),
 Stuff(a=25, b=1518),
 Stuff(a=281, b=501),
 Stuff(a=228, b=285),
 Stuff(a=754, b=209),
 Stuff(a=692, b=1516),
 Stuff(a=913, b=1116),
 Stuff(a=89, b=1209),
 Stuff(a=432, b=65),
 Stuff(a=30, b=191)]

**Algorithm 1: Full-blown Aggregation**

In [2]:
import math


def aggregate(input_data, attribute):
    """Full-blown aggregation algorithm: computes an aggregation over a list of tuples, once it is done, it returns the result"""
    current_max = -math.inf
    for el in input_data:
        current_max = max(current_max, el.__dict__[attribute])
    return current_max


aggregate(data, "b")

1888

**Algorithm 2: Online Aggregation**

In [3]:
import math


def online_aggregation(input_data, attribute):
    """Online aggregation algorithm: computes a running aggregation over a list of tuples yielding the new result after each tuple"""
    current_max = -math.inf
    for el in input_data:
        current_max = max(current_max, el.__dict__[attribute])
        yield current_max


for agg in online_aggregation(data, "b"):
    print(agg)

228
1518
1518
1518
1518
1518
1518
1518
1518
1518
1518
1518
1518
1518
1518
1518
1518
1518
1780
1780
1780
1780
1780
1780
1780
1780
1780
1780
1780
1780
1780
1780
1780
1888
1888
1888
1888
1888
1888
1888


In [4]:
import unittest


class MyTest(unittest.TestCase):

    # the following function will be called before every test-function:

    def create_data(self, number_of_tuples):
        self.data = [Stuff(fake.pyint(max_value=20), fake.pyint(max_value=20)) for f in range(number_of_tuples)]

    def test_scans_end_to_end(self):
        for i in range(10):
            self.create_data(1000)

            res1 = aggregate(self.data, "b")
            *_, res2 = online_aggregation(self.data, "b")

            self.assertEqual(res1, res2)


# Run the unit test without shutting down the jupyter kernel
# notice that this call will look for all test classes, i.e.
# all classes inheriting from unittest.TestCase
# each of these functions will be executed once
# unittest.main(argv=['ignored', '-v'], verbosity=2, exit=False)

# only execute a specific Test-class:
unittest.main(argv=['ignored', '-v'], defaultTest='MyTest', verbosity=2, exit=False)


test_scans_end_to_end (__main__.MyTest.test_scans_end_to_end) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.021s

OK
